# Inferencia de las 30 clases en BLD-AR

> Inferencia por tiles con SAHI de las 30 clases de FloorPlanCAD sobre los 58 planos de BLD-AR, para revisar a ojo qué predice el modelo. Muestra un plano y guarda los 58 en un .zip.
>
> **Incluye clases no etiquetadas en BLD-AR:** solo 6 de las 30 tienen etiqueta equivalente (`docs/MAPEO_CLASES.md`). Las otras 24 (muebles, sanitarios, electrodomésticos, etc.) no tienen con qué compararse: sus predicciones no se miden, solo se miran.

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8

## Importar librerías
> SAHI para inferir por tiles, matplotlib para dibujar las máscaras y zipfile para guardar las figuras.

In [ ]:
import json
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch, Polygon
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import read_image_as_pil

## Definir rutas y parámetros
> Corrida `01-YOLO26l-SEG-1024` y los mismos parámetros de SAHI que `evaluate_bldar.ipynb`, para ver las mismas predicciones que se evalúan ahí.

In [ ]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
BLDAR = REPO / "datasets/BLD-AR"
WEIGHTS = REPO / "weights/01-YOLO26l-SEG-1024/best.pt"
OUT_ZIP = REPO / "runs" / WEIGHTS.parent.name / "bld-ar-unlabeled-too.zip"
PLAN = sorted(BLDAR.glob("*.json"))[3]  # JSON LabelMe del plano que se muestra en el notebook

TILE = 1024
OVERLAP = 0.2
CONF = 0.05
VIS_CONF = 0.25  # no se dibujan las predicciones con menos confianza
SAVE_DPI = 300  # resolución de las figuras del .zip, para ver los objetos chicos
DEVICE = "cuda:0"

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Leer las etiquetas de un plano
> Solo las 6 clases de `docs/MAPEO_CLASES.md`, con el nombre de FloorPlanCAD para que coincidan con las predicciones.

In [ ]:
# Clase de FloorPlanCAD (predicción) -> clase de BLD-AR (etiqueta)
CLASS_MAP = {
    "wall": "wall", "single_door": "door", "window": "window", "sliding_door": "sliding_door",
    "stair": "stair", "elevator": "elevator",
}
to_floorplancad = {bldar: floorplancad for floorplancad, bldar in CLASS_MAP.items()}


def shape_points(shape):
    """Puntos de una forma LabelMe; los círculos se aproximan con 32 lados."""
    points = np.array(shape["points"], dtype=float)
    if shape["shape_type"] == "circle":
        center, edge = points
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        points = center + np.linalg.norm(edge - center) * np.stack([np.cos(angles), np.sin(angles)], axis=1)
    return points


def read_labels(data):
    """(clase, puntos) de las etiquetas de las 6 clases mapeadas en un JSON LabelMe."""
    return [(to_floorplancad[shape["label"]], shape_points(shape))
            for shape in data["shapes"] if shape["label"] in to_floorplancad]

## Cargar el modelo
> SAHI necesita la corrida envuelta en su propio modelo para inferir por tiles.

In [ ]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)

## Definir la inferencia por tiles
> Igual que `evaluate_bldar.ipynb`, pero sin filtrar por el mapeo: quedan las predicciones de las 30 clases.

In [ ]:
def predict(image_path):
    """(clase, puntos) de las predicciones de un plano con confianza >= VIS_CONF."""
    result = get_sliced_prediction(
        str(image_path), model,
        slice_height=TILE, slice_width=TILE,
        overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        perform_standard_pred=False, verbose=0,
        # Con confianza < 0.1 SAHI cambia a NMS y no une los muros cortados entre tiles
        force_postprocess_type=True,
    )
    return [(p["category_name"], np.reshape(polygon, (-1, 2)))
            for p in result.to_coco_predictions() if p["score"] >= VIS_CONF for polygon in p["segmentation"]]

## Definir la figura de un plano
> Etiqueta (6 clases) y predicción (30 clases) lado a lado, como en `compare_evaluation_methods.ipynb`.

> Máscaras rellenas y semitransparentes, un color fijo por clase; las 6 mapeadas llevan los colores de `compare_evaluation_methods.ipynb`. La leyenda muestra solo las clases dibujadas en el plano.

In [ ]:
# Las 6 clases mapeadas con tab10; las otras 24, con tab20b y tab20c
others = [name for name in model.category_names if name not in CLASS_MAP]
colors = dict(zip([*CLASS_MAP, *others], plt.cm.tab10.colors[:6] + plt.cm.tab20b.colors + plt.cm.tab20c.colors))


def draw(image_path, labels, predicted):
    """Etiqueta y predicción lado a lado, con las máscaras rellenas por clase."""
    picture = read_image_as_pil(str(image_path)).convert("L")
    width, height = picture.size
    picture.thumbnail((4000, 4000))  # hay planos de hasta 15000 px de lado
    fig, axes = plt.subplots(1, 2, figsize=(24, 12 * height / width), gridspec_kw={"wspace": 0.02})
    for ax, shapes, title in zip(axes, (labels, predicted), ("Etiqueta (6 clases)", "Predicción (30 clases)")):
        # extent ubica la imagen reducida en las coordenadas originales
        ax.imshow(picture, cmap="gray", extent=(0, width, height, 0))
        for name, points in shapes:
            ax.add_patch(Polygon(points, color=colors[name], alpha=0.4))
        ax.set_title(f"{title} · {Path(image_path).name}")
        ax.axis("off")
    drawn = {name for name, _ in labels + predicted}
    fig.legend(handles=[Patch(color=color, label=name) for name, color in colors.items() if name in drawn],
               loc="upper center", ncol=10)
    return fig

## Guardar los 58 planos en un .zip y mostrar uno
> En la celda la figura se achica al ancho del notebook; el .zip permite bajarlas todas juntas y mirarlas en alta resolución.

In [ ]:
OUT_ZIP.parent.mkdir(parents=True, exist_ok=True)
with ZipFile(OUT_ZIP, "w") as archive:
    for json_path in sorted(BLDAR.glob("*.json")):
        data = json.loads(json_path.read_text())
        fig = draw(BLDAR / data["imagePath"], read_labels(data), predict(BLDAR / data["imagePath"]))
        image = BytesIO()
        fig.savefig(image, format="jpg", dpi=SAVE_DPI, bbox_inches="tight", pil_kwargs={"quality": 95})
        # En una carpeta dentro del .zip, para que al descomprimirlo queden juntas
        archive.writestr(f"{OUT_ZIP.stem}/{json_path.stem}.jpg", image.getvalue())
        if json_path == PLAN:
            plt.show()
        else:
            plt.close(fig)